In [1]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import os, pickle

import numpy as np
import matplotlib, matplotlib.pyplot as plt
from matplotlib import ticker
import hist
import vector
import pandas as pd

from physics.simulation import msq, mcfm
from physics.hstar import c6
from datasets import coefficient
from models import taylr

import torch
from torch.utils.data import DataLoader
import lightning as L

In [ ]:
torch.set_float32_matmul_precision('high')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [3]:
RUN_DIR = 'run/h4l'
TAYLR_DIR = 'taylr'

CHECKPOINTS = [ '30', '21', '25', '194' ]
LOSSES = ['0.08', '0.15', '0.08', '0.05']
VERSIONS = ['0', '0', '0', '0']

ncoeffs = len(CHECKPOINTS)

EVENTS_TRAIN_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}','events_train.pkl') for i  in range(ncoeffs)]
EVENTS_VAL_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}','events_val.pkl') for i  in range(ncoeffs)]
EVENTS_TEST_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'events_test.pkl') for i  in range(ncoeffs)]

SCALER_X_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'scaler_X.pkl') for i in range(ncoeffs)]
SCALER_y_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'scaler_y.pkl') for i in range(ncoeffs)]

METRICS_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'lightning_logs', f'version_{VERSIONS[i]}', 'metrics.csv') for i in range(ncoeffs)]
CHECKPOINT_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'lightning_logs', f'version_{VERSIONS[i]}', 'checkpoints', f'epoch={CHECKPOINTS[i]}-val_loss={LOSSES[i]}.ckpt') for i in range(ncoeffs)]

In [4]:
TEST_SIZE = 1200000
BATCH_SIZE = 1024
SEED = 42
FEATURES = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

In [5]:
with open(EVENTS_TRAIN_PATHS[0], 'rb') as f:
    events_train = pickle.load(f)
with open(EVENTS_VAL_PATHS[0], 'rb') as f:
    events_val = pickle.load(f)
with open(EVENTS_TEST_PATHS[0], 'rb') as f:
    events_test = pickle.load(f)

In [6]:
for i in range(ncoeffs):
    with open(SCALER_X_PATHS[i], 'rb') as f:
        scaler_X = pickle.load(f)
    validation_datasets = [coefficient.CoefficientDataset(events_val, features=FEATURES, coefficient_index=i+1, component=msq.Component.SBI, scaler_X=scaler_X) for i in range(ncoeffs)]
    testing_datasets = [coefficient.CoefficientDataset(events_test, features=FEATURES, coefficient_index=i+1, component=msq.Component.SBI, scaler_X=scaler_X) for i in range(ncoeffs)]

In [7]:
metrics = [np.genfromtxt(path, delimiter=',') for path in METRICS_PATHS]

epochs = [m[1::2,0] for m in metrics]
steps = [m[1::2,1] for m in metrics]
train_loss = [m[0::2,2][1:] for m in metrics]
val_loss = [m[1::2,3] for m in metrics]

In [8]:
fig, axes = plt.subplots(2,2, figsize=(8,5), layout='constrained')

for i in range(ncoeffs):
    axes[i//2,i%2].plot(epochs[i], train_loss[i], color='red', label='training loss')
    axes[i//2,i%2].plot(epochs[i], val_loss[i], color='royalblue', label='testing loss')

    axes[i//2,i%2].legend(frameon=False, fontsize=12)
    axes[i//2,i%2].set_xlabel('training epoch', fontsize=12)
    axes[i//2,i%2].set_ylabel('loss', fontsize=12)
    axes[i//2,i%2].tick_params(axis='both', labelsize=12)

plt.savefig('taylr_loss.pdf', bbox_inches='tight')

In [ ]:
models = [taylr.TAYLR.load_from_checkpoint(path) for path in CHECKPOINT_PATHS]

In [ ]:
validation_dataloaders = [DataLoader(val, BATCH_SIZE) for val in validation_datasets]
testing_dataloaders = [DataLoader(val, BATCH_SIZE) for val in testing_datasets]

In [ ]:
trainer = L.Trainer(accelerator='gpu')
pred_val = [torch.concatenate(trainer.predict(model, testing_dataloaders[i])).numpy()[:,np.newaxis].flatten() for i, model in enumerate(models)]
pred_test = [torch.concatenate(trainer.predict(model, testing_dataloaders[i])).numpy()[:,np.newaxis].flatten() for i, model in enumerate(models)]

target_val = [ds.y for ds in validation_datasets]
target_test = [ds.y for ds in testing_datasets]

weight_val = [ds.w for ds in validation_datasets]
weight_test = [ds.w for ds in testing_datasets]

for i in range(ncoeffs):
    with open(SCALER_y_PATHS[i], 'rb') as f:
        scaler_y = pickle.load(f)

    pred_val[i] = scaler_y.inverse_transform(pred_val[i].reshape(-1,1)).reshape(-1)
    pred_test[i] = scaler_y.inverse_transform(pred_test[i].reshape(-1,1)).reshape(-1)

Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
l1 = vector.array({"pt": events_test.kinematics['l1_pt'], "eta": events_test.kinematics['l1_eta'], "phi": events_test.kinematics['l1_phi'], "energy": events_test.kinematics['l1_energy']})
l2 = vector.array({"pt": events_test.kinematics['l2_pt'], "eta": events_test.kinematics['l2_eta'], "phi": events_test.kinematics['l2_phi'], "energy": events_test.kinematics['l2_energy']})
l3 = vector.array({"pt": events_test.kinematics['l3_pt'], "eta": events_test.kinematics['l3_eta'], "phi": events_test.kinematics['l3_phi'], "energy": events_test.kinematics['l3_energy']})
l4 = vector.array({"pt": events_test.kinematics['l4_pt'], "eta": events_test.kinematics['l4_eta'], "phi": events_test.kinematics['l4_phi'], "energy": events_test.kinematics['l4_energy']})

m4l = (l1 + l2 + l3 + l4).mass

In [ ]:
c6_val = -20
c6_mod = c6.Modifier(baseline=msq.Component.SBI, events=events_test, c6_values=[-5,-1,0,1,5])
w_c6, p_c6 = [w.flatten() for w in c6_mod.modify(c6_val)]

In [ ]:
coefficients = np.concatenate([np.ones_like(p_c6)[:,np.newaxis], pred_test[0][:,np.newaxis], pred_test[1][:,np.newaxis], pred_test[2][:,np.newaxis], pred_test[3][:,np.newaxis]], axis=1)
reweight = np.apply_along_axis(lambda x: np.polyval(x, c6_val), 1, coefficients[:, ::-1])

In [ ]:
bins = 41
bounds = [180,1000]

lw = 1.25

h_true = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_true.fill(m4l, weight=p_c6)

p_SM = events_test.weights / events_test.weights.sum()
h_SM = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_SM.fill(m4l, weight=p_SM)

p_rwt = p_SM * reweight / np.sum(p_SM * reweight)
h_reweight = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_reweight.fill(m4l, weight=p_rwt)

fig, (ax1, ax2, ax3) = plt.subplots(3,1,gridspec_kw={'height_ratios': [3, 1, 1]},figsize=(4,6), layout='constrained', sharex=True)
plt.subplots_adjust(wspace=0, hspace=0)

ax1.stairs(h_true.values(), h_true.axes[0].edges, color='royalblue', linestyle='-', linewidth=lw, alpha=0.8, label=f'$c_6 = {c6_val}$')
ax1.stairs(h_SM.values(), h_SM.axes[0].edges, color='black', linestyle='--', linewidth=lw, alpha=0.8, label='$\\mathrm{SM}$')
ax1.stairs(h_reweight.values(), h_reweight.axes[0].edges, color='red',  linestyle='-', linewidth=lw, alpha=0.8, label=f'$\\mathrm{{SM}} \\rightarrow\\ (c_6 = {c6_val})\\ \\mathrm{{(NSBI)}}$')

ax1.set_xlabel('')
ax1.set_ylabel('$\\mathrm{Fraction\\ of\\ events}$', loc='top')
ax1.set_xlim(180,1000)

ax1.set_yscale('log')
ax1.legend(frameon=False)

ax2.errorbar(h_true.axes[0].centers, h_true.values()/h_SM.values(), yerr=np.sqrt(h_true.variances())/h_SM.values(), color='royalblue', linewidth=lw, elinewidth=lw, drawstyle='steps-mid', label=f'Truth qqZZ', alpha=0.8)
ax2.errorbar(h_reweight.axes[0].centers, h_reweight.values()/h_SM.values(), yerr=np.sqrt(h_reweight.variances())/h_SM.values(), color='r', linewidth=lw, elinewidth=lw, drawstyle='steps-mid', label=f'CARL ggZZ->qqZZ')

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)
ax3.tick_params(labelsize=12)

ax2.set_ylabel('$p(x | c_6) / p_{ \\mathrm{SM} }(x)$')
ax2.set_xlim(180,1000)
ax2.set_ylim(0.9,1.15)

ax2.set_xlabel('')
ax2.set_xticklabels([])

ax3.errorbar(h_true.axes[0].centers, h_true.values()/h_true.values(), yerr=np.sqrt(h_true.variances())/h_true.values(), color='royalblue', linewidth=lw, elinewidth=lw, drawstyle='steps-mid', label=f'Truth qqZZ', alpha=0.8)
ax3.errorbar(h_reweight.axes[0].centers, h_reweight.values()/h_true.values(), yerr=np.sqrt(h_reweight.variances())/h_true.values(), color='r', linewidth=lw, elinewidth=lw, drawstyle='steps-mid', label=f'CARL ggZZ->qqZZ')

ax3.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right')
ax3.set_xlim(180,1000)
ax3.set_xscale('linear')

ax3.set_ylabel('$\\mathrm{NSBI} / \\mathrm{MC}$', loc='center')
ax3.set_ylim(0.95,1.05)

plt.subplots_adjust(hspace=0)
plt.tight_layout()
plt.savefig('taylr_reweight.pdf', bbox_inches='tight')